# Quickstart

A bivariate choropleth shows two variables at once: each has an axis of colours, and the mix of both says where they are high together, low together, or apart.

In [ ]:
%pip install bivario basemaplibre folium geopandas lonboard matplotlib

In [ ]:
import basemaplibre as bml
import geopandas as gpd
import matplotlib.pyplot as plt
from mapclassify import Quantiles

from bivario import explore_bivariate_data, get_bivariate_cmap, plot_bivariate_legend
from bivario.example_data import nyc_bike_trips

## An interactive map

The example data has bike trips in New York, counted in H3 cells. Where do people start their morning trips, and where do they end them?

In [ ]:
trips = nyc_bike_trips()
trips.head()

In [ ]:
explore_bivariate_data(trips, "morning_starts", "morning_ends")

Values are split into five classes by natural breaks. Labels, the number of classes, the palette and the dark mode are all parameters:

In [ ]:
explore_bivariate_data(
    trips,
    "morning_starts",
    "morning_ends",
    column_a_label="Starts",
    column_b_label="Ends",
    scheme="Quantiles",
    k=3,
    cmap="bubblegum",
    dark_mode=True,
)

More of it in the [Folium](folium_example.ipynb) and [Lonboard](lonboard_example.ipynb) examples.

## Your own data

Any GeoDataFrame with two columns of numbers works. Here are the countries of Europe from [Natural Earth](https://www.naturalearthdata.com/): how densely populated they are, and how rich.

In [ ]:
url = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/"
    "ne_50m_admin_0_countries.geojson"
)
countries = gpd.read_file(url, columns=["NAME", "CONTINENT", "POP_EST", "GDP_MD"])
countries = countries[countries["POP_EST"] > 100_000]
europe = countries[countries["CONTINENT"] == "Europe"].to_crs(3035)
europe["density"] = europe["POP_EST"] / (europe.area / 1e6)
europe["gdp_per_capita"] = europe["GDP_MD"] * 1e6 / europe["POP_EST"]
europe.sort_values("gdp_per_capita", ascending=False).head()

## A static map

A colourmap takes the two columns and returns a colour for every row, which is what `GeoDataFrame.plot` needs. Here the values are first classified into three groups of the same size:

In [ ]:
density = Quantiles(europe["density"], k=3)
gdp = Quantiles(europe["gdp_per_capita"], k=3)

cmap = get_bivariate_cmap("plum_mint")
colours = cmap(density.yb, gdp.yb)
colours[:3]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9), dpi=150)
ax.set_axis_off()
ax.set_xlim(1_900_000, 6_000_000)
ax.set_ylim(1_400_000, 4_700_000)

europe.plot(ax=ax, color=colours, alpha=0.85, edgecolor="white", linewidth=0.5)
bml.add_basemap(ax, crs=europe.crs, source="OpenFreeMap.Positron", labels="above")

plot_bivariate_legend(
    europe["density"],
    europe["gdp_per_capita"],
    ax=ax.inset_axes((0.07, 0.74, 0.22, 0.22)),
    cmap=cmap,
    grid_size=3,
    label_a="People per km²",
    label_b="GDP per capita [$]",
    tick_labels_a=[f"{value:,.0f}" for value in (europe["density"].min(), *density.bins)],
    tick_labels_b=[f"{value:,.0f}" for value in (europe["gdp_per_capita"].min(), *gdp.bins)],
    tick_fontsize_px=15,
)
plt.show()

The basemap is drawn by [basemaplibre](https://raczeq.github.io/basemaplibre/) in the CRS of the data, with its labels above the countries. The [Matplotlib](matplotlib_example.ipynb) example shows other layouts, the [colourmaps](colourmaps.ipynb) one how to make your own palette.